# AlloFusion Model Comparison

Compare baseline vs topology-augmented models with publication-ready figures.

---

## 📖 Quick Start Guide

### Step 1: Train Baseline Model
1. Open `cnn_train_with_tuner.ipynb`
2. In **Cell 26** (Topology Augmentation), set:
   ```python
   USE_DCI_BETWEENNESS = False  # Baseline: 1047D features
   ```
3. Run all cells → trains model, saves to `myModel/`
4. Rename output directory:
   ```bash
   mv myModel myModel_baseline
   ```

### Step 2: Train Topology Model
1. In `cnn_train_with_tuner.ipynb`, **Cell 26**, set:
   ```python
   USE_DCI_BETWEENNESS = True  # Topology: 1050D features
   ```
2. Run all cells → trains model, saves to `myModel/`
3. Rename output directory:
   ```bash
   mv myModel myModel_topology
   ```

### Step 3: Compare Models
1. Open this notebook (`model_comparison.ipynb`)
2. Run all cells
3. View comparison plots inline
4. Outputs saved to `results/comparison/`

---

## 📊 What This Notebook Generates

### 1. **Publication Figure** (Panel A + B)
- **Panel A**: ROC curves overlaid (Baseline vs Topology)
- **Panel B**: Performance bar chart (SEN, PRE, MCC, F1, AUC)
- Format: PNG (300 DPI) + PDF (for LaTeX)
- File: `publication_comparison.png` / `.pdf`

### 2. **Extended Metrics Chart**
- All 7 metrics: SEN, SPE, PRE, MCC, F1, AUC, AUPRC
- Side-by-side comparison with value labels
- File: `all_metrics_comparison.png`

### 3. **Improvement Chart**
- Horizontal bars showing % improvement per metric
- File: `improvement_chart.png`

### 4. **Console Output**
- Performance comparison table
- Feature dimensions (1047D vs 1050D)
- Best epoch info
- Statistical significance

---

## 📁 Expected Directory Structure

```
AlloFusion-main/
├── myModel_baseline/
│   ├── training_metrics.json      ← Test performance
│   ├── training_history.json      ← Training curves
│   ├── test_predictions.csv       ← Per-residue scores
│   └── final_model_1047d.h5       ← Model weights
├── myModel_topology/
│   ├── training_metrics.json
│   ├── training_history.json
│   ├── test_predictions.csv
│   └── final_model_1050d.h5
├── results/
│   └── comparison/
│       ├── publication_comparison.png  ← Main figure
│       ├── publication_comparison.pdf
│       ├── all_metrics_comparison.png
│       └── improvement_chart.png
├── cnn_train_with_tuner.ipynb    ← Training notebook
└── model_comparison.ipynb         ← This notebook
```

---

## 🎯 Key Metrics Explained

| Metric | Full Name | Description |
|--------|-----------|-------------|
| **SEN** | Sensitivity (Recall) | % of true allosteric residues correctly identified |
| **SPE** | Specificity | % of non-allosteric residues correctly rejected |
| **PRE** | Precision | % of predicted allosteric residues that are correct |
| **MCC** | Matthews Correlation Coefficient | Balanced measure (-1 to +1) |
| **F1** | F1-Score | Harmonic mean of precision and recall |
| **AUC** | AUROC | Area under ROC curve |
| **AUPRC** | Area Under PR Curve | Best metric for imbalanced data |

---

## ⚠️ Troubleshooting

**Error: "FileNotFoundError: myModel_baseline/training_metrics.json"**
- Make sure you trained the baseline model first
- Check that you renamed `myModel/` to `myModel_baseline/`

**Error: "Feature dimensions don't match"**
- Baseline should be 1047D (T5 + PSSM + Bio)
- Topology should be 1050D (+ DCI + Betweenness)
- Check `USE_DCI_BETWEENNESS` flag in training notebook

**Plots not displaying?**
- Make sure you're running in Jupyter (not VSCode Python Interactive)
- Check that matplotlib backend is set correctly

---

## 🚀 Ready to Compare?

Run the cells below to generate all comparison figures!

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import roc_curve, roc_auc_score

# Set style for publication
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 9

print('✓ Setup complete')

In [ ]:
# Configure paths
BASELINE_DIR = Path('myModel_baseline')
TOPOLOGY_DIR = Path('myModel_topology')
OUTPUT_DIR = Path('results/comparison')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load results
def load_results(model_dir):
    with open(model_dir / 'training_metrics.json') as f:
        metrics = json.load(f)
    predictions = pd.read_csv(model_dir / 'test_predictions.csv')
    return metrics, predictions

baseline_metrics, baseline_preds = load_results(BASELINE_DIR)
topology_metrics, topology_preds = load_results(TOPOLOGY_DIR)

print(f'✓ Baseline: {len(baseline_preds):,} samples')
print(f'✓ Topology: {len(topology_preds):,} samples')

## Publication Figure: ROC + Performance Metrics

In [ ]:
# Create figure with two subplots
fig = plt.figure(figsize=(14, 6))
gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.2], hspace=0.3, wspace=0.3)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])

# ========== LEFT PANEL: ROC CURVES ==========

# Baseline ROC
fpr_base, tpr_base, _ = roc_curve(baseline_preds['label'], baseline_preds['score'])
auc_base = roc_auc_score(baseline_preds['label'], baseline_preds['score'])

# Topology ROC
fpr_topo, tpr_topo, _ = roc_curve(topology_preds['label'], topology_preds['score'])
auc_topo = roc_auc_score(topology_preds['label'], topology_preds['score'])

# Define colors matching the paper style
colors = {
    'baseline': '#4472C4',  # Blue
    'topology': '#70AD47',  # Green
}

# Plot ROC curves
ax1.plot(fpr_base, tpr_base, color=colors['baseline'], lw=2.5, 
         label=f'Baseline (AUC = {auc_base:.3f})')
ax1.plot(fpr_topo, tpr_topo, color=colors['topology'], lw=2.5, 
         label=f'Baseline+Topology (AUC = {auc_topo:.3f})')

# Diagonal line
ax1.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.5, label='No Skill (AUC = 0.500)')

# Styling
ax1.set_xlabel('False Positive Rate', fontweight='bold')
ax1.set_ylabel('True Positive Rate', fontweight='bold')
ax1.set_title('ROC curve of Characteristic', fontweight='bold')
ax1.set_xlim([0, 1])
ax1.set_ylim([0, 1.02])
ax1.legend(loc='lower right', frameon=True, fancybox=True, shadow=False)
ax1.grid(True, alpha=0.2, linestyle='-', linewidth=0.5)
ax1.text(-0.15, 1.05, 'A', transform=ax1.transAxes, fontsize=16, fontweight='bold')

# ========== RIGHT PANEL: PERFORMANCE METRICS ==========

metrics_order = ['SEN', 'PRE', 'MCC', 'F1', 'AUC']
metric_map = {
    'SEN': 'sensitivity',
    'PRE': 'precision_at_threshold',
    'MCC': 'mcc',
    'F1': 'f1_score',
    'AUC': 'test_auc'
}

baseline_vals = [baseline_metrics[metric_map[m]] for m in metrics_order]
topology_vals = [topology_metrics[metric_map[m]] for m in metrics_order]

x = np.arange(len(metrics_order))
width = 0.35

# Create bars
bars1 = ax2.bar(x - width/2, baseline_vals, width, 
                label='Baseline', 
                color=colors['baseline'], 
                alpha=0.8, 
                edgecolor='black', 
                linewidth=0.8)

bars2 = ax2.bar(x + width/2, topology_vals, width, 
                label='Baseline+Topology', 
                color=colors['topology'], 
                alpha=0.8, 
                edgecolor='black', 
                linewidth=0.8)

# Add value labels on bars
def add_value_labels(bars):
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=8.5, fontweight='bold')

add_value_labels(bars1)
add_value_labels(bars2)

# Styling
ax2.set_ylabel('Performance', fontweight='bold')
ax2.set_ylim([0, 1.05])
ax2.set_xticks(x)
ax2.set_xticklabels(metrics_order, fontweight='bold')
ax2.legend(loc='upper left', frameon=True, fancybox=True, shadow=False)
ax2.grid(axis='y', alpha=0.2, linestyle='-', linewidth=0.5)
ax2.text(-0.15, 1.05, 'B', transform=ax2.transAxes, fontsize=16, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'publication_comparison.png', dpi=300, bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'publication_comparison.pdf', bbox_inches='tight')  # For LaTeX
print(f'✓ Publication figure saved to: {OUTPUT_DIR}/publication_comparison.png')
print(f'✓ PDF version saved to: {OUTPUT_DIR}/publication_comparison.pdf')
plt.show()

## Print Summary Statistics

In [ ]:
print('\n' + '='*80)
print('PERFORMANCE COMPARISON')
print('='*80)
print(f'{"Metric":<12} {"Baseline":<12} {"Topology":<12} {"Δ":<10} {"% Change":<10}')
print('-'*80)

for metric_name, metric_key in metric_map.items():
    base_val = baseline_metrics[metric_key]
    topo_val = topology_metrics[metric_key]
    diff = topo_val - base_val
    pct = (diff / base_val * 100) if base_val != 0 else 0
    
    print(f'{metric_name:<12} {base_val:<12.4f} {topo_val:<12.4f} {diff:<10.4f} {pct:<10.2f}%')

print('='*80)
print(f'\nFeature Dimensions:')
print(f'  Baseline: {baseline_metrics["feature_dim"]}D (T5 + PSSM + Bio)')
print(f'  Topology: {topology_metrics["feature_dim"]}D (T5 + PSSM + Bio + DCI + Betweenness)')
print(f'\nBest Epochs:')
print(f'  Baseline: {baseline_metrics["best_epoch"]}/{baseline_metrics["total_epochs_run"]}')
print(f'  Topology: {topology_metrics["best_epoch"]}/{topology_metrics["total_epochs_run"]}')

## Additional Comparison Plots

In [ ]:
# Overlay all metrics in one bar chart
fig, ax = plt.subplots(figsize=(10, 6))

all_metrics = ['SEN', 'SPE', 'PRE', 'MCC', 'F1', 'AUC', 'AUPRC']
all_metric_map = {
    'SEN': 'sensitivity',
    'SPE': 'specificity',
    'PRE': 'precision_at_threshold',
    'MCC': 'mcc',
    'F1': 'f1_score',
    'AUC': 'test_auc',
    'AUPRC': 'auprc'
}

baseline_all = [baseline_metrics[all_metric_map[m]] for m in all_metrics]
topology_all = [topology_metrics[all_metric_map[m]] for m in all_metrics]

x = np.arange(len(all_metrics))
width = 0.35

bars1 = ax.bar(x - width/2, baseline_all, width, 
               label='Baseline (1047D)', 
               color=colors['baseline'], 
               alpha=0.8, 
               edgecolor='black', 
               linewidth=0.8)

bars2 = ax.bar(x + width/2, topology_all, width, 
               label='Baseline+Topology (1050D)', 
               color=colors['topology'], 
               alpha=0.8, 
               edgecolor='black', 
               linewidth=0.8)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
               f'{height:.3f}',
               ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Metric', fontweight='bold', fontsize=12)
ax.set_ylabel('Score', fontweight='bold', fontsize=12)
ax.set_title('Comprehensive Performance Comparison', fontweight='bold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(all_metrics, fontweight='bold')
ax.set_ylim([0, 1.05])
ax.legend(loc='lower right', frameon=True, shadow=True)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'all_metrics_comparison.png', dpi=300, bbox_inches='tight')
print(f'✓ All metrics comparison saved to: {OUTPUT_DIR}/all_metrics_comparison.png')
plt.show()

## Improvement Heatmap

In [ ]:
# Create improvement matrix
improvement_data = []
for metric_name, metric_key in all_metric_map.items():
    base_val = baseline_metrics[metric_key]
    topo_val = topology_metrics[metric_key]
    improvement = ((topo_val - base_val) / base_val * 100) if base_val != 0 else 0
    improvement_data.append({'Metric': metric_name, 'Improvement (%)': improvement})

df_improvement = pd.DataFrame(improvement_data)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(df_improvement['Metric'], df_improvement['Improvement (%)'], 
               color=[colors['topology'] if x > 0 else 'red' for x in df_improvement['Improvement (%)']], 
               alpha=0.8, edgecolor='black', linewidth=0.8)

# Add value labels
for i, (metric, val) in enumerate(zip(df_improvement['Metric'], df_improvement['Improvement (%)'])):
    ax.text(val + 0.2 if val > 0 else val - 0.2, i, 
            f'{val:+.2f}%', 
            va='center', ha='left' if val > 0 else 'right', 
            fontweight='bold', fontsize=9)

ax.set_xlabel('Improvement (%)', fontweight='bold', fontsize=12)
ax.set_ylabel('Metric', fontweight='bold', fontsize=12)
ax.set_title('Performance Improvement with Topology Features', fontweight='bold', fontsize=14)
ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
ax.grid(axis='x', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'improvement_chart.png', dpi=300, bbox_inches='tight')
print(f'✓ Improvement chart saved to: {OUTPUT_DIR}/improvement_chart.png')
plt.show()

print('\n' + '='*80)
print('All comparison figures generated successfully!')
print('='*80)